In [1]:
import json
import operator
import random
import torch

from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import login

from help_lib import is_valid_equation, extract_last_answer, extract_generated_equation

In [ ]:
TEACHER_MODEL = "Qwen/Qwen2.5-7B-Instruct"
NUM_SAMPLES_1_DIGIT = 1000
NUM_SAMPLES_3_DIGITS = 3000
OUTPUT_FILE = "raw_teacher_countdown.jsonl"
BATCH_SIZE = 16

SYSTEM_CONTENT = ("You are a helpful assistant. You first think about the reasoning process "
                  "in the mind and then provide the user with the answer.")



login()

device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
def generate_problem(target_type="3-digit"):
    ops = {
        "+": operator.add,
        "-": operator.sub,
        "*": operator.mul,
        "/": operator.floordiv,
    }
    
    while True:
        nums = [random.randint(1, 100) for _ in range(random.choice([3, 4, 5]))]
        orig_nums = nums.copy()
        curr = nums.copy()
        random.shuffle(curr)
        try:
            for _ in range(len(curr) - 1):
                a, b = curr.pop(), curr.pop()
                op = random.choice(list(ops.keys()))
                
                if op == "/" and (b == 0 or a % b != 0):
                    op = "+"
                if op == "-" and a < b:
                    a, b = b, a
                curr.append(ops[op](a, b))

            target = curr[0]
            if target_type == "1-digit" and 0 <= target <= 9:
                return orig_nums, target
            if target_type == "3-digit" and 100 <= target <= 999:
                return orig_nums, target
        except:
            continue


def get_user_content(nums, target):
        return (f"Using the numbers {nums}, create an equation that equals {target}. "
                "You can use basic arithmetic operations (+, -, *, /) and each number can only be used once. " 
                "Show your work in <think> </think> tags. And return the final equation and answer in "
                "<answer> </answer> tags, for example <answer> (1 + 2) / 3 = 1 </answer>.")

In [ ]:
def main():
    tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL)
    tokenizer.padding_side = "left"
    model = AutoModelForCausalLM.from_pretrained(
        TEACHER_MODEL, 
        dtype=torch.bfloat16, 
        device_map="auto"
    )

    problems = []
    for _ in range(NUM_SAMPLES_1_DIGIT):
        problems.append(generate_problem("1-digit"))
    for _ in range(NUM_SAMPLES_3_DIGITS):
        problems.append(generate_problem("3-digit"))
    
    random.shuffle(problems)
    

    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        
        for i in tqdm(range(0, len(problems), BATCH_SIZE)):
            batch = problems[i : i + BATCH_SIZE]
            prompts = []
            for nums, target in batch:
                msgs = [
                    {"role": "system", "content": SYSTEM_CONTENT},
                    {"role": "user", "content": get_user_content(nums, target)},
                ]
                
                prompts.append(
                    tokenizer.apply_chat_template(
                        msgs, 
                        tokenize=False, 
                        add_generation_prompt=True
                    )
                )

            inputs = tokenizer(prompts, padding=True, return_tensors="pt").to(device)

            with torch.no_grad():
                outputs = model.generate(
                    **inputs, 
                    max_new_tokens=512, 
                    temperature=0.6, 
                    do_sample=True
                )

            for j, out_tokens in enumerate(outputs):
                nums, target = batch[j]
                
                full_text = tokenizer.decode(
                    out_tokens[inputs.input_ids.shape[1] :],
                    skip_special_tokens=True
                )
                
                equation = extract_generated_equation(extract_last_answer(full_text))
                if is_valid_equation(equation, nums, target):
                    entry = {
                        "target": target,
                        "nums": nums,
                        "messages": [
                            {"role": "system", "content": SYSTEM_CONTENT},
                            {"role": "user", "content": get_user_content(nums, target)},
                            {"role": "assistant", "content": full_text.strip()},
                        ],
                    }
                    
                    f.write(json.dumps(entry, ensure_ascii=False) + "\n")

    print(f"Data is saved in {OUTPUT_FILE}")

In [ ]:
main()